# A Tiny Residual and a Useless Answer

**Topic:** Conditioning &nbsp;·&nbsp; **Difficulty:** Medium

You solve `Ax = b`, check the residual `||Ax - b||`, see `1e-11`, and declare
victory. On an ill-conditioned system that check proves almost nothing: the
residual can be at machine precision while every digit of `x` is wrong.

The condition number bounds the conversion rate between the two. Measure it.

## The data (created in the next cell)

- `A`: a 12x12 Hilbert-like matrix, notoriously ill-conditioned.
- `x_true`: the exact solution, all ones.
- `b = A @ x_true`, then perturbed by a relative `1e-10`.

## Your task

With **NumPy only**:

1. Solve `A x = b_pert` with `np.linalg.solve`.
2. `residual`: `||A x - b_pert||_2 / ||b_pert||_2`.
3. `fwd_error`: `||x - x_true||_2 / ||x_true||_2`.
4. `kappa`: the 2-norm condition number of `A` (`np.linalg.cond`).
5. `amplification`: `fwd_error / relative_perturbation`, where the relative
   perturbation is `||b_pert - b||_2 / ||b||_2`.
6. Set `answers` to a dict with `"residual"`, `"fwd_error"`, `"kappa"`,
   `"amplification"`, and `"bound_holds"`: `1.0` if
   `amplification <= kappa * 1.0001`, else `0.0`.

## Evaluation

The forward error, the condition number and the amplification are compared to the
reference and must land within 50% of it: on this matrix they shift by a few
percent from one NumPy build to the next. The evaluation cell also checks the
residual is tiny (under 1e-10) while the forward error is not (over 1e-4), and
that the amplification respects the condition-number bound.

In [ ]:
import numpy as np

n = 12
i = np.arange(1, n + 1)
A = 1.0 / (i[:, None] + i[None, :] - 1.0)      # Hilbert matrix
x_true = np.ones(n)
b = A @ x_true

rng = np.random.default_rng(3003)
delta = rng.normal(0, 1, n)
delta = delta / np.linalg.norm(delta) * (1e-10 * np.linalg.norm(b))
b_pert = b + delta
print(f"A is {n}x{n} Hilbert, perturbing b by a relative 1e-10")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("residual", [], 1.0665582030324573e-11, -1.7858238819987418e-12, 1e-06),
    ("fwd_error", [], 503908.40367518435, -84373.42275971614, 0.5),
    ("kappa", [], 1.7253251431391934e+16, -2888850168767681.5, 0.5),
    ("amplification", [], 5039084095495803.0, -843734237433133.4, 0.5),
    ("bound_holds", [], 1.0, -0.16743801481449916, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "residual" in answers:
    _small = answers["residual"] < 1e-10
    _big = answers["fwd_error"] > 1e-4
    print(f"  {'residual tiny':<24} {'ok' if _small else 'wrong'}  ({answers['residual']:.2e})")
    print(f"  {'forward error large':<24} {'ok' if _big else 'wrong'}  ({answers['fwd_error']:.2e})")
    print(f"  {'bound respected':<24} {'ok' if answers['bound_holds'] == 1.0 else 'wrong'}")
    _ok = _ok and _small and _big and answers["bound_holds"] == 1.0

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

The bound is `relative forward error <= kappa * relative backward error`. For a
12x12 Hilbert matrix `kappa` is around `1e16`, so a `1e-10` perturbation in `b`
licenses a forward error of order `1e6`. The answer can be arbitrarily wrong
while the residual stays at machine precision, because the residual measures
*backward* error only. This is why "my residual is small" is not a correctness
argument, and why you report `cond` alongside any linear solve.

```python
import numpy as np

x = np.linalg.solve(A, b_pert)

residual = float(np.linalg.norm(A @ x - b_pert) / np.linalg.norm(b_pert))
fwd_error = float(np.linalg.norm(x - x_true) / np.linalg.norm(x_true))
kappa = float(np.linalg.cond(A))
rel_pert = float(np.linalg.norm(b_pert - b) / np.linalg.norm(b))
amplification = fwd_error / rel_pert

answers = {
    "residual": residual,
    "fwd_error": fwd_error,
    "kappa": kappa,
    "amplification": amplification,
    "bound_holds": float(amplification <= kappa * 1.0001),
}
```